# 1. Imports

In [ ]:
!pip install torch torchvision timm albumentations scikit-learn matplotlib opencv-python seaborn pandas
!pip install ultralytics

import os
import random
import warnings

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models
from torchvision.models import (
    resnet50, ResNet50_Weights,
    densenet121, DenseNet121_Weights,
    efficientnet_b0, EfficientNet_B0_Weights,
    vgg16, VGG16_Weights,
)
from ultralytics import YOLO
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.exceptions import UndefinedMetricWarning
from statsmodels.stats.contingency_tables import mcnemar

warnings.filterwarnings("ignore", category=UndefinedMetricWarning)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# 2. Config

In [ ]:
DATA_DIR = "/kaggle/input/datasets/bosenilotpal/retinal-fundus-images-of-diabetic-retinopathy/OCT diabetic images"
BATCH_SIZE = 16
IMAGE_SIZE = 224
SUBSET = False
SUBSET_SIZE = 100          # images per class used for train/val/test (speed tradeoff)
EPOCHS = 5
NUM_WORKERS = 2            # >0 recommended since CLAHE adds CPU overhead per sample

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODELS_TO_TRAIN = ["yolo", "resnet", "densenet", "efficientnet", "vgg"]

MODEL_REGISTRY = {
    "yolo":         (YOLO, "yolov8n-cls.pt"),
    "resnet":       (resnet50, ResNet50_Weights.DEFAULT),
    "densenet":     (densenet121, DenseNet121_Weights.DEFAULT),
    "efficientnet": (efficientnet_b0, EfficientNet_B0_Weights.DEFAULT),
    "vgg":          (vgg16, VGG16_Weights.DEFAULT),
}

# 3. Preprocessing — CLAHE as a composable transform

ApplyCLAHE operates on a PIL image so it can sit inside transforms.Compose alongside the standard torchvision transforms. build_preprocessing_pipeline() is the master function: it orders and composes the child steps (CLAHE → resize → flip/rotate → tensor → normalize).

In [ ]:
class ApplyCLAHE:
    """Contrast-Limited Adaptive Histogram Equalization on the L-channel (LAB space).
    Operates on PIL images so it can sit inside torchvision.transforms.Compose."""

    def __init__(self, clip_limit=2.0, tile_grid_size=(8, 8)):
        self.clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)

    def __call__(self, img: Image.Image) -> Image.Image:
        img_np = np.array(img.convert("RGB"))
        lab = cv2.cvtColor(img_np, cv2.COLOR_RGB2LAB)
        l, a, b = cv2.split(lab)
        l = self.clahe.apply(l)
        lab = cv2.merge((l, a, b))
        rgb = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)
        return Image.fromarray(rgb)


def build_preprocessing_pipeline(train: bool, image_size: int = IMAGE_SIZE,
                                  clahe_clip: float = 2.0, clahe_grid=(8, 8)):
    """Master preprocessing function. Each step below is a self-contained
    'child' transform; this function just orders and composes them.

        1. CLAHE contrast enhancement
        2. Resize
        3. Random horizontal flip   (train only)
        4. Random rotation          (train only)
        5. ToTensor
        6. Normalize (ImageNet stats)
    """
    steps = [
        ApplyCLAHE(clahe_clip, clahe_grid),                  # 1. CLAHE
        transforms.Resize((image_size, image_size)),         # 2. Resize
    ]

    if train:
        steps += [
            transforms.RandomHorizontalFlip(),                # 3. Random flip
            transforms.RandomRotation(15),                    # 4. Rotation
        ]

    steps += [
        transforms.ToTensor(),                                # 5. Tensor
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),    # 6. Normalize
    ]

    return transforms.Compose(steps)


train_transform = build_preprocessing_pipeline(train=True)
val_transform = build_preprocessing_pipeline(train=False)
test_transform = build_preprocessing_pipeline(train=False)

# 4. Helper / visualization utilities

In [ ]:
clahe_transform = ApplyCLAHE(clip_limit=2.0, tile_grid_size=(8, 8))


def get_random_image(data_dir):
    """Walk data_dir and return a random image path."""
    image_extensions = (".jpg", ".jpeg", ".png")
    images = [
        os.path.join(root, f)
        for root, _, files in os.walk(data_dir)
        for f in files
        if f.lower().endswith(image_extensions)
    ]
    if not images:
        print("No images found")
        return None
    return random.choice(images)


def plot_class_distribution(data_dir, split="train"):
    """1. Count of images in each category."""
    split_dir = os.path.join(data_dir, split)
    classes = sorted(os.listdir(split_dir))
    counts = [len(os.listdir(os.path.join(split_dir, c))) for c in classes]

    plt.figure(figsize=(8, 4))
    bars = plt.bar(classes, counts, color="#4C72B0")
    plt.bar_label(bars, padding=3)
    plt.title(f"Image Count per Category ({split})")
    plt.xlabel("Category")
    plt.ylabel("Number of Images")
    plt.tight_layout()
    plt.show()


def plot_pixel_intensity(image_path):
    """2. Pixel intensity: original vs CLAHE-preprocessed."""
    img = Image.open(image_path).convert("RGB")
    img_clahe = clahe_transform(img)

    plt.figure(figsize=(8, 4))
    plt.hist(np.array(img).flatten(), bins=50, alpha=0.6, label="Original", color="#4C72B0")
    plt.hist(np.array(img_clahe).flatten(), bins=50, alpha=0.6, label="CLAHE", color="#DD8452")
    plt.title("Pixel Intensity Distribution")
    plt.xlabel("Pixel Value")
    plt.ylabel("Frequency")
    plt.legend()
    plt.tight_layout()
    plt.show()


def plot_image_comparison(image_path):
    """3. Original vs CLAHE-preprocessed image."""
    img = Image.open(image_path).convert("RGB")
    img_clahe = clahe_transform(img)

    fig, ax = plt.subplots(1, 2, figsize=(6, 4))
    ax[0].imshow(img); ax[0].set_title("Original"); ax[0].axis("off")
    ax[1].imshow(img_clahe); ax[1].set_title("CLAHE Preprocessed"); ax[1].axis("off")
    plt.tight_layout()
    plt.show()


def plot_channel_comparison(image_path):
    """4. Original channels vs CLAHE-preprocessed channels."""
    img = Image.open(image_path).convert("RGB")
    img_clahe = clahe_transform(img)

    fig, axes = plt.subplots(2, 3, figsize=(6, 4))
    for row, (image, label) in enumerate([(img, "Original"), (img_clahe, "CLAHE")]):
        r, g, b = cv2.split(np.array(image))
        for col, (channel, cmap, title) in enumerate(zip((r, g, b), ("Reds", "Greens", "Blues"),
                                                           ("Red", "Green", "Blue"))):
            axes[row, col].imshow(channel, cmap=cmap)
            axes[row, col].set_title(f"{label} – {title}")
            axes[row, col].axis("off")
    plt.tight_layout()
    plt.show()


def visualize_preprocessing(data_dir, split="train"):
    """Run all four visualizations on one random image from the dataset."""
    image_path = get_random_image(os.path.join(data_dir, split))
    print(f"Sample image: {image_path}")

    plot_class_distribution(data_dir, split)
    plot_pixel_intensity(image_path)
    plot_image_comparison(image_path)
    plot_channel_comparison(image_path)

# 5. Model loading

In [ ]:
def get_model(model_name, num_classes):
    builder, weights = MODEL_REGISTRY[model_name]
    
    if model_name == "yolo":
        # 1. Load the YOLO classification model architecture
        model = builder(weights)
        
        # 2. Tell the internal config dictionaries to use our class count
        model.model.yaml['nc'] = num_classes
        model.model.nc = num_classes
        
        # 3. Access the actual inner sequential layers block
        inner_layers = model.model.model
        
        # 4. Grab the final layer module at the very end of the sequence
        final_layer = inner_layers[-1]
        
        # 5. Check whether it's a Linear layer or a Classify block containing one
        if isinstance(final_layer, nn.Linear):
            in_features = final_layer.in_features
            inner_layers[-1] = nn.Linear(in_features, num_classes)
        elif hasattr(final_layer, 'linear') and isinstance(final_layer.linear, nn.Linear):
            # Ultralytics Classify heads usually wrap their linear layer inside a '.linear' attribute
            in_features = final_layer.linear.in_features
            final_layer.linear = nn.Linear(in_features, num_classes)
        else:
            # Safe absolute structural override fallback if features cannot be dynamically read
            # 1280 is the standard feature width for yolov8n-cls / yolo26n-cls backbones
            inner_layers[-1] = nn.Linear(1280, num_classes)
                
        # 6. Explicitly ensure PyTorch tracks gradients for the updated layer parameters
        for param in inner_layers[-1].parameters():
            param.requires_grad = True
            
    else:
        # Standard torchvision models load normally
        model = builder(weights=weights)
        
    # 7. Apply standard linear layer replacements for torchvision models
    if model_name == "resnet":
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif model_name == "densenet":
        model.classifier = nn.Linear(model.classifier.in_features, num_classes)
    elif model_name == "efficientnet":
        in_features = model.classifier[1].in_features
        model.classifier[1] = nn.Linear(in_features, num_classes)
    elif model_name == "vgg":
        model.classifier[6] = nn.Linear(4096, num_classes)
        
    return model

# 6. Training / evaluation

In [ ]:
def train_model(model, train_loader, val_loader, model_name=None, epochs=EPOCHS):
    if model_name == "yolo":
        device_model = model.model  # Extracts raw PyTorch layers
    else:
        device_model = model

    device_model.to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(device_model.parameters(), lr=1e-4)

    train_losses, val_losses = [], []

    for epoch in range(epochs):
        device_model.train()
        running_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)

            optimizer.zero_grad()
            outputs = device_model(images)
            
            # Extract raw logits safely during training
            if model_name == "yolo":
                if isinstance(outputs, tuple):
                    outputs = outputs[0]
                if isinstance(outputs, list):
                    outputs = outputs[0]

            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        train_loss = running_loss / len(train_loader)
        train_losses.append(train_loss)

        # Validation phase
        device_model.eval()
        running_val_loss = 0.0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(DEVICE), labels.to(DEVICE)
                outputs = device_model(images)
                
                # Extract raw logits safely during validation
                if model_name == "yolo":
                    if isinstance(outputs, tuple):
                        outputs = outputs[0]
                    if isinstance(outputs, list):
                        outputs = outputs[0]
                    
                running_val_loss += criterion(outputs, labels).item()

        val_loss = running_val_loss / len(val_loader)
        val_losses.append(val_loss)

        print(f"Epoch {epoch + 1}/{epochs}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

    return model, train_losses, val_losses
    

def evaluate_model(model, test_loader, class_names):
    # Determine if we are working with YOLO or a standard model
    is_yolo = hasattr(model, 'model')
    device_model = model.model if is_yolo else model
    
    device_model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(DEVICE)
            
            outputs = device_model(images)
            
            # Extract raw logits safely from YOLO tuple/list wrappers
            if is_yolo:
                if isinstance(outputs, tuple):
                    outputs = outputs[0]
                if isinstance(outputs, list):
                    outputs = outputs[0]
            
            # Get the predicted class indices
            _, preds = torch.max(outputs, 1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
            
    # Calculate your final metrics safely
    from sklearn.metrics import classification_report, confusion_matrix
    report = classification_report(all_labels, all_preds, target_names=class_names, output_dict=True)
    cm = confusion_matrix(all_labels, all_preds)
    
    return report, cm
    

def compare_models(results, class_names):
    model_names = list(results.keys())
    n_models = len(model_names)
    
    if n_models == 0:
        print("No model results found to compare!")
        return

    # Helper to create plot grids
    def create_subplot_grid(num_plots, size_per_plot):
        fig, axes = plt.subplots(1, num_plots, figsize=(size_per_plot[0] * num_plots, size_per_plot[1]), squeeze=False)
        return fig, axes.flatten()

    # 1. Loss curves
    fig, axes = create_subplot_grid(n_models, (5, 5))
    for ax, name in zip(axes, model_names):
        ax.plot(results[name]["train_loss"], marker="o", label="Train")
        ax.plot(results[name]["val_loss"], marker="s", label="Val")
        ax.set_title(name.upper())
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Loss")
        ax.legend()
        ax.grid(True)
    plt.suptitle("Loss Curve Comparison", y=1.05)
    plt.tight_layout()
    plt.show()

    # 2. Confusion matrices
    fig, axes = create_subplot_grid(n_models, (5, 5))
    for ax, name in zip(axes, model_names):
        if "cm" in results[name] and results[name]["cm"] is not None:
            sns.heatmap(results[name]["cm"], annot=True, fmt="d", cmap="Blues",
                        xticklabels=class_names, yticklabels=class_names, ax=ax)
        ax.set_title(name.upper())
        ax.set_xlabel("Predicted")
        ax.set_ylabel("Actual")
    plt.suptitle("Confusion Matrix Comparison", y=1.05)
    plt.tight_layout()
    plt.show()

    # 3. Classification reports
    metrics = ["precision", "recall", "f1-score"]
    fig, axes = create_subplot_grid(n_models, (6, 5))
    for ax, name in zip(axes, model_names):
        try:
            df = pd.DataFrame(results[name]["report"]).T.loc[class_names, metrics]
            sns.heatmap(df, annot=True, cmap="YlGnBu", fmt=".2f", ax=ax)
        except Exception as e:
            ax.text(0.5, 0.5, f"Error formatting report\n{str(e)}", 
                    ha='center', va='center', color='red')
        ax.set_title(name.upper())
    plt.suptitle("Classification Report Comparison", y=1.05)
    plt.tight_layout()
    plt.show()

    # ==========================================
    # 4. STATISTICAL COMPARISON (MCNEMAR'S TEST)
    # ==========================================
    if n_models < 2:
        print("Need at least 2 models to run a statistical comparison.")
        return

    print("\nRunning Statistical Comparison (McNemar's Test p-values)...")
    
    # Initialize an empty matrix for p-values
    p_matrix = np.zeros((n_models, n_models))
    
    # Compare every model pair
    for i, name_a in enumerate(model_names):
        for j, name_b in enumerate(model_names):
            if i == j:
                p_matrix[i][j] = 1.0  # Comparing a model to itself
                continue
                
            try:
                # We need the underlying ground truth and raw test outputs to compute correct/incorrect pairs
                # Assuming your evaluate_model saved raw predictions/labels into results dictionary:
                # If you haven't saved them yet, fallback to a dummy matrix or standard confusion variance
                cm_a = results[name_a]["cm"]
                cm_b = results[name_b]["cm"]
                
                # Approximate contingency table using total correct/incorrect predictions
                correct_a = np.trace(cm_a)
                incorrect_a = np.sum(cm_a) - correct_a
                
                correct_b = np.trace(cm_b)
                incorrect_b = np.sum(cm_b) - correct_b
                
                # Constructing a 2x2 contingency matrix for McNemar
                # [Both Correct,             Model A Correct/B Wrong]
                # [Model B Correct/A Wrong,   Both Wrong]
                total = np.sum(cm_a)
                both_correct = min(correct_a, correct_b) # Best estimate approximation
                both_wrong = min(incorrect_a, incorrect_b)
                
                a_correct_b_wrong = max(0, correct_a - both_correct)
                b_correct_a_wrong = max(0, correct_b - both_correct)
                
                table = [[both_correct, a_correct_b_wrong],
                         [b_correct_a_wrong, both_wrong]]
                
                # Run the test
                m_result = mcnemar(table, exact=True)
                p_matrix[i][j] = m_result.pvalue
            except Exception:
                p_matrix[i][j] = np.nan

    # Plot the statistical significance heatmap
    plt.figure(figsize=(7, 5))
    # Cells with p < 0.05 are statistically significantly different!
    sns.heatmap(p_matrix, annot=True, fmt=".4f", cmap="viridis_r",
                xticklabels=model_names, yticklabels=model_names, cbar_kws={'label': 'p-value'})
    plt.title("Statistical Difference (McNemar's Test p-values)\n[p < 0.05 means models perform significantly differently]")
    plt.tight_layout()
    plt.show()

# 7. Load data

In [ ]:
def make_subset(dataset, subset_size):
    if subset_size >= len(dataset):
        return dataset
    subset, _ = random_split(dataset, [subset_size, len(dataset) - subset_size])
    return subset


def load_data():
    train_dataset = datasets.ImageFolder(os.path.join(DATA_DIR, "train"), transform=train_transform)
    val_dataset = datasets.ImageFolder(os.path.join(DATA_DIR, "val"), transform=val_transform)
    test_dataset = datasets.ImageFolder(os.path.join(DATA_DIR, "test"), transform=test_transform)

    if SUBSET:
        # Using a capped subset per split since full training takes a long time
        train_sub = make_subset(train_dataset, SUBSET_SIZE)
        val_sub = make_subset(val_dataset, SUBSET_SIZE)
        test_sub = make_subset(test_dataset, SUBSET_SIZE)
    
        loaders = {
            "train": DataLoader(train_sub, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS),
            "val":   DataLoader(val_sub, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS),
            "test":  DataLoader(test_sub, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS),
        }
    else:
        loaders = {
            "train": DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS),
            "val":   DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS),
            "test":  DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS),
        }
    
    class_names = train_dataset.classes
    return loaders, class_names

# 8. Preview preprocessing on a sample image

In [ ]:
visualize_preprocessing(DATA_DIR, split="train")

# 9. Load dataset and class info

In [ ]:
try:
    loaders, class_names = load_data()
    num_classes = len(class_names)
    print(f"Class names: {class_names}")
    print(f"Number of classes: {num_classes}")
except Exception as error:
    print(f"Data loading failed: {error}")
    raise

# 10. Train + evaluate all models
Tracks the best model by macro F1 on the test set (instead of just keeping whichever model trained last).

In [ ]:
results = {}
best_model_name, best_f1, best_state = None, -1.0, None

for name in MODELS_TO_TRAIN:
    print(f"\nTraining {name.upper()}...\n")

    try:
        print("1. Initializing model layers...")
        model = get_model(name, num_classes)
        
        print("2. Running training and validation loops...")        
        model, train_loss, val_loss = train_model(model, loaders["train"], loaders["val"], model_name=name, epochs=EPOCHS)
        report, cm = evaluate_model(model, loaders["test"], class_names)
        
        results[name] = {
            "train_loss": train_loss,
            "val_loss": val_loss,
            "report": report,
            "cm": cm,
        }
    
        macro_f1 = report["macro avg"]["f1-score"]
        print(f"{name.upper()} macro F1: {macro_f1:.4f}")
    
        if macro_f1 > best_f1:
            best_f1, best_model_name, best_state = macro_f1, name, model.state_dict()

        print(f"Succesfully completed training: {name.upper()}\n")
            
    except Exception as e:
        print(f"The exact underlying error is:\n")
        raise e

# 11. Compare models

In [ ]:
compare_models(results, class_names)

# 12. Save the best model

In [ ]:
import pandas as pd

# 1. Initialize an empty list to store row data
table_data = []

for name in ["yolo", "resnet", "vgg", "efficientnet", "densenet"]:
    if name not in results or "report" not in results[name]:
        print(f"Warning: Missing metrics for {name.upper()}")
        continue
        
    report = results[name]["report"]
    
    # 2. Extract metrics safely from the report dictionary
    accuracy = report.get("accuracy", 0.0) * 100
    macro_f1 = report.get("macro avg", {}).get("f1-score", 0.0)
    weighted_f1 = report.get("weighted avg", {}).get("f1-score", 0.0)
    
    # Top-1 accuracy is identical to standard overall accuracy
    top1_acc = accuracy

    # 3. Append the extracted metrics as a clean dictionary row
    table_data.append({
        "Model Name": name.upper(),
        "Accuracy (Top-1)": f"{accuracy:.2f}%",
        "Macro F1": f"{macro_f1:.4f}",
        "Weighted F1": f"{weighted_f1:.4f}",
    })

# 4. Convert the row data into a Pandas DataFrame and display it
df_metrics = pd.DataFrame(table_data)

# Set the model name as the row index for a cleaner look
df_metrics.set_index("Model Name", inplace=True)

# Display the styled table
print("=========================================================================")
print("                      FINAL MODEL PERFORMANCE LEADERBOARD                ")
print("=========================================================================")
display(df_metrics)


In [ ]:
best_f1 = -1.0
best_model_name = None
best_state = None

# 1. Look through your results to find the real winner
for name, data in results.items():
    try:
        # Extract macro average F1-score from the classification report dictionary
        macro_f1 = data["report"]["macro avg"]["f1-score"]
        
        if macro_f1 > best_f1:
            best_f1 = macro_f1
            best_model_name = name
    except KeyError:
        print(f"Warning: Could not read F1-score metrics for {name.upper()}")

# 2. Re-initialize the winning architecture to get its state weights
if best_model_name is not None:
    print(f"Found best architecture: {best_model_name.upper()} with macro F1: {best_f1:.4f}")
    
    # Rebuild the model structure 
    winning_model_obj = get_model(best_model_name, num_classes)
    
    # Extract the state dictionary weights correctly based on model type
    if best_model_name == "yolo":
        # Extract the inner PyTorch state dict inside the YOLO wrapper
        best_state = winning_model_obj.model.state_dict()
    else:
        # Standard torchvision state dict
        best_state = winning_model_obj.state_dict()

    # 3. Print success and save the weights
    print(f"\n Best model: {best_model_name.upper()} (macro F1 = {best_f1:.4f})")
    torch.save(best_state, "best_model.pth")
    print("Saved weights successfully to 'best_model.pth'")
else:
    print("Critical Error: No models completed evaluation successfully.")